# Stage 3 Revised — Berth State, Berth Entry, and Predicted Release Inputs

Tujuan tahap ini adalah menyediakan keadaan operasional yang diperlukan Stage 4:
waktu mulai sandar, lama sandar berjalan, perkiraan sisa pelayanan, dan perkiraan
waktu dermaga dilepas oleh kapal yang sedang menempatinya.

In [3]:
from pathlib import Path
import os, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

def locate_root():
    candidates = [
        Path(os.environ.get("MFAR_PROJECT_ROOT", "")),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
        Path.cwd(), Path.cwd().parent
    ]
    for p in candidates:
        if str(p) and (p/"stage_output").exists() and (p/"config").exists():
            return p.resolve()
    raise FileNotFoundError("MFAR_Modular_Colab_Pipeline tidak ditemukan.")

ROOT = locate_root()
RAW = ROOT/"data_raw"
CFG = ROOT/"config"
STAGE = ROOT/"stage_output"
for i in range(1,8):
    (STAGE/f"stage_{i:02d}").mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline


In [4]:
INFILE = STAGE/"stage_02"/"02_vessel_interpolated_grid.csv"
if not INFILE.exists():
    raise FileNotFoundError(
        f"Output Stage 2 tidak ditemukan: {INFILE}. Jalankan Stage 2 terlebih dahulu."
    )

state = pd.read_csv(INFILE)
state["grid_time"] = pd.to_datetime(state["grid_time"], errors="coerce")
state = state.dropna(subset=["grid_time","mmsi"]).sort_values(["mmsi","grid_time"])

profiles = pd.read_csv(CFG/"vessel_profiles.csv")
berths = pd.read_csv(CFG/"terminal_berths.csv")

for col in ["turnaround_min","approach_allowance_min","vehicle_capacity_ce","normal_sog_kn"]:
    if col not in state.columns:
        state = state.merge(profiles[["mmsi",col]], on="mmsi", how="left")

In [5]:
# Normalisasi status sandar.
state["is_at_berth"] = state["operational_status"].astype(str).str.startswith("AT_BERTH")
state["berth_episode_start"] = False

for mmsi, idx in state.groupby("mmsi").groups.items():
    idx = list(idx)
    prev = False
    for i in idx:
        now = bool(state.loc[i,"is_at_berth"])
        state.loc[i,"berth_episode_start"] = now and not prev
        prev = now

# Nomor episode sandar per kapal.
state["berth_episode_id"] = (
    state.groupby("mmsi")["berth_episode_start"].cumsum()
)

episode_start = (
    state[state["is_at_berth"]]
    .groupby(["mmsi","berth_episode_id"])["grid_time"]
    .transform("min")
)
state.loc[state["is_at_berth"],"berth_entry_time"] = episode_start
state["berth_entry_time"] = pd.to_datetime(state["berth_entry_time"], errors="coerce")

state["elapsed_berth_min"] = np.where(
    state["is_at_berth"],
    (state["grid_time"]-state["berth_entry_time"]).dt.total_seconds()/60,
    0.0
)

# turnaround_min dipakai sebagai estimasi total pelayanan di dermaga.
state["predicted_remaining_service_min"] = np.where(
    state["is_at_berth"],
    np.maximum(
        0.0,
        pd.to_numeric(state["turnaround_min"], errors="coerce").fillna(40.0)
        - state["elapsed_berth_min"]
    ),
    0.0
)

# Tambahkan allowance manuver keluar 5 menit.
DEPARTURE_MANEUVER_MIN = 5.0
state["predicted_berth_release_time"] = pd.NaT
mask = state["is_at_berth"]
state.loc[mask,"predicted_berth_release_time"] = (
    state.loc[mask,"grid_time"]
    + pd.to_timedelta(
        state.loc[mask,"predicted_remaining_service_min"] + DEPARTURE_MANEUVER_MIN,
        unit="m"
    )
)

# Dermaga yang sedang ditempati.
state["occupied_berth_id"] = np.where(
    state["is_at_berth"],
    state["current_berth_id"],
    np.nan
)

In [6]:
S3 = STAGE/"stage_03"
state.to_csv(S3/"03_input_state_enhanced.csv", index=False)

release = state.loc[state["is_at_berth"],[
    "grid_time","mmsi","vessel_name","origin","occupied_berth_id",
    "berth_entry_time","elapsed_berth_min","predicted_remaining_service_min",
    "predicted_berth_release_time","turnaround_min"
]].copy()
release.to_csv(S3/"03_predicted_berth_release_state.csv", index=False)

audit = pd.DataFrame({
    "check":[
        "missing_release_time_for_occupied_berth",
        "negative_elapsed_berth",
        "negative_remaining_service",
        "duplicate_vessel_time"
    ],
    "failed_rows":[
        int(state.loc[mask,"predicted_berth_release_time"].isna().sum()),
        int((state["elapsed_berth_min"]<0).sum()),
        int((state["predicted_remaining_service_min"]<0).sum()),
        int(state.duplicated(["grid_time","mmsi"]).sum())
    ]
})
audit.to_csv(S3/"03_berth_prediction_audit.csv", index=False)
display(audit)

,check,failed_rows
0,missing_release_time_for_occupied_berth,0
1,negative_elapsed_berth,0
2,negative_remaining_service,0
3,duplicate_vessel_time,0
